# RO-054 Planning with motion limits: every number in the Note

The models live in `images/motionkit.py` (the robot, its limits, the room, the simulator, the lattice search and the
kinodynamic RRT) and `images/carpaths.py` (Dubins and Reeds-Shepp paths). This notebook runs each one and prints the
numbers the Note quotes. The random parts (the reachable-set trails, the RRT) use fixed seeds.

In [1]:
import sys, math
sys.path.insert(0, 'images')
import numpy as np
import motionkit as m
import carpaths as cp

## 1. The limits

In [2]:
print('largest turn rate at 0.5 m/s:', m.W_MAX, 'rad/s; tightest circle:', m.RHO, 'm')
print('wheel speeds at v = 0.5, omega = 1:', m.wheel_speeds(0.5, 1.0))
for v in (0.5, 0.6):
    print(f'from {v} m/s: stops after {v / m.A_MAX:.1f} s and {m.stopping_distance(v):.2f} m')
print('the line to the doorway:', round(m.LINE_LEN, 3), 'm at', round(math.degrees(m.LINE_HEADING), 1), 'degrees')
print('the disk meets the closed door at s =', round(m.S_DOOR, 4), 'm')
v_hit = math.sqrt(0.6 ** 2 - 2 * m.A_MAX * (m.S_DOOR - 3.0))
print('braking from 0.6 m/s at s = 3.0 m: meets the door at', round(v_hit, 3), 'm/s')
print('swing at a 45-degree corner turned at 1 rad/s:', round(m.RHO * (1 - math.cos(math.pi / 4)), 4), 'm')

largest turn rate at 0.5 m/s: 1.0 rad/s; tightest circle: 0.5 m
wheel speeds at v = 0.5, omega = 1: (0.6, 0.4)
from 0.5 m/s: stops after 1.0 s and 0.25 m
from 0.6 m/s: stops after 1.2 s and 0.36 m
the line to the doorway: 3.4 m at 61.9 degrees
the disk meets the closed door at s = 3.1167 m
braking from 0.6 m/s at s = 3.0 m: meets the door at 0.493 m/s
swing at a 45-degree corner turned at 1 rad/s: 0.1464 m


## 2. Two robots at s = 2.8 m

In [3]:
for v0 in (0.3, 0.6):
    t, s, v = m.brake_run(2.8, v0)
    print(f'{v0} m/s: ends at s = {s[-1]:.4f} m with {v[-1]:.4f} m/s after {t[-1]:.3f} s', 'HIT' if v[-1] > 0 else 'stopped')
print('room left:', round(m.S_DOOR - 2.8, 4))

0.3 m/s: ends at s = 2.8900 m with 0.0000 m/s after 0.600 s stopped
0.6 m/s: ends at s = 3.1167 m with 0.2082 m/s after 0.784 s HIT
room left: 0.3167


## 3. The region of inevitable collision

In [4]:
for v in (0.2, 0.4, 0.5, 0.6):
    print(f'v = {v}: stopping distance {m.stopping_distance(v):.2f} m, doomed beyond s = {m.S_DOOR - m.stopping_distance(v):.4f} m')
print('A (2.8, 0.3) doomed?', m.inevitable(2.8, 0.3), '  B (2.8, 0.6) doomed?', m.inevitable(2.8, 0.6))

v = 0.2: stopping distance 0.04 m, doomed beyond s = 3.0767 m
v = 0.4: stopping distance 0.16 m, doomed beyond s = 2.9567 m
v = 0.5: stopping distance 0.25 m, doomed beyond s = 2.8667 m
v = 0.6: stopping distance 0.36 m, doomed beyond s = 2.7567 m
A (2.8, 0.3) doomed? False   B (2.8, 0.6) doomed? True


## 4. The reachable set

In [5]:
th = m.LINE_HEADING
t_face = (th - m.START[2]) / m.W_MAX
x = m.START[0] + m.RHO * (math.sin(th) - math.sin(m.START[2]))
y = m.START[1] - m.RHO * (math.cos(th) - math.cos(m.START[2]))
print('turn needed:', round(th - m.START[2], 4), 'rad; first faces 61.9 deg after', round(t_face, 4), 's, having driven',
      round(m.V_CRUISE * t_face, 4), 'm, at', (round(x, 4), round(y, 4)))
trails = m.reach_trails(T=1.5, n=400, seed=1, h=0.02)
print('trails drawn:', len(trails))

turn needed: 0.5572 rad; first faces 61.9 deg after 0.5572 s, having driven 0.2786 m, at (2.1912, 1.1977)
trails drawn: 403


## 5. Motion primitives and the simulator

In [6]:
tr = m.simulate(m.START, 0.5, 1.0, 0.01)
print('one Euler step of the left primitive:', tr[-1])
for name, om in m.PRIMS.items():
    e = m.simulate(m.START, m.V_CRUISE, om, m.DT_PRIM)[-1]
    exact = cp.step(m.START, {'left': 'L', 'straight': 'S', 'right': 'R'}[name], m.V_CRUISE * m.DT_PRIM, m.RHO)
    print(f'{name:9s} end ({e[0]:.4f}, {e[1]:.4f}) at {math.degrees(e[2]):.2f} deg; exact arc ({exact[0]:.4f}, {exact[1]:.4f})')
print('each primitive drives', m.V_CRUISE * m.DT_PRIM, 'm and turns', round(math.degrees(m.W_MAX * m.DT_PRIM), 2), 'deg')

one Euler step of the left primitive: [2.00433013 1.0025     0.53359878]
left      end (2.2295, 1.2854) at 72.97 deg; exact arc (2.2281, 1.2866)
straight  end (2.3248, 1.1875) at 30.00 deg; exact arc (2.3248, 1.1875)
right     end (2.3620, 1.0560) at -12.97 deg; exact arc (2.3622, 1.0542)
each primitive drives 0.375 m and turns 42.97 deg


## 6. Lattice search on the straight drive

In [7]:
for dt in (0.5, 0.25, 0.2, 0.1):
    r = m.lattice_search(dt)
    print(f'dt = {dt}: speed step {r["dv"]:.3f} m/s, position step {r["q"]:.5f} m, {r["steps"]} steps = {r["time"]:.1f} s, {r["n_states"]} states')
r = m.lattice_search(0.5)
print('plan:', r['path'])
print('least time with the true limits (speed up, cruise at 0.6, brake):', round(m.bang_coast_bang(), 3), 's')

dt = 0.5: speed step 0.250 m/s, position step 0.06250 m, 14 steps = 7.0 s, 73 states
dt = 0.25: speed step 0.125 m/s, position step 0.01562 m, 28 steps = 7.0 s, 483 states
dt = 0.2: speed step 0.100 m/s, position step 0.01000 m, 31 steps = 6.2 s, 1043 states
dt = 0.1: speed step 0.050 m/s, position step 0.00250 m, 62 steps = 6.2 s, 7749 states
plan: [(0.0, 0.0), (0.0625, 0.25), (0.25, 0.5), (0.5, 0.5), (0.75, 0.5), (1.0, 0.5), (1.25, 0.5), (1.5, 0.5), (1.75, 0.5), (2.0, 0.5), (2.25, 0.5), (2.5, 0.5), (2.75, 0.5), (2.9375, 0.25), (3.0, 0.0)]
least time with the true limits (speed up, cruise at 0.6, brake): 6.2 s


## 7. Dubins and Reeds-Shepp paths

In [8]:
words = cp.dubins_words(m.START, m.DOOR_POSE, m.RHO)
for w, p in words.items():
    print(w, round(cp.length(p), 4), [(k, round(s, 4)) for k, s in p])
c0, c1 = cp.circles(m.START, m.RHO), cp.circles(m.DOOR_POSE, m.RHO)
print('left circles', round(math.dist(c0['L'], c1['L']), 4), 'm apart; right circles', round(math.dist(c0['R'], c1['R']), 4))
w, p = cp.dubins(m.START, m.DOOR_POSE, m.RHO)
pts = cp.sample(m.START, p, m.RHO, ds=0.01)
print('shortest', w, '; worst clearance', round(min(m.clearance(a, b) for a, b, _ in pts), 4), 'm')
rs = cp.reeds_shepp(m.START, m.DOOR_POSE, m.RHO)
print('Reeds-Shepp to the door pose:', cp.word(rs), round(cp.length(rs), 4))

LSL 2.9924 [('L', 0.2343), ('S', 2.4688), ('L', 0.2893)]
RSR 9.2273 [('R', 2.8994), ('S', 3.4677), ('R', 2.8602)]
LSR 6.1067 [('L', 0.2615), ('S', 2.9656), ('R', 2.8795)]
RSL 6.119 [('R', 2.9258), ('S', 2.8853), ('L', 0.3079)]
left circles 2.4688 m apart; right circles 3.4677
shortest LSL ; worst clearance -0.1417 m
Reeds-Shepp to the door pose: L+ S+ L+ 2.9924


In [9]:
S = m.START
G = (S[0] - 0.5 * math.sin(S[2]), S[1] + 0.5 * math.cos(S[2]), S[2])
rs = cp.reeds_shepp(S, G, m.RHO); w, d = cp.dubins(S, G, m.RHO)
print('sideways 0.5 m to', tuple(round(v, 4) for v in G))
print('  Reeds-Shepp', cp.word(rs), round(cp.length(rs), 4), 'm, reversals', cp.cusps(rs), [round(s, 4) for _, s in rs])
print('  Dubins', w, round(cp.length(d), 4), 'm', [round(s, 4) for _, s in d])
print('  times at 0.5 m/s:', round(cp.length(rs) / 0.5, 3), round(cp.length(d) / 0.5, 3), ' ratio', round(cp.length(d) / cp.length(rs), 3))
print('  RO-038 distance between the two poses:', m.metric(S, G))
back = (S[0] - 0.5 * math.cos(S[2]), S[1] - 0.5 * math.sin(S[2]), S[2])
print('0.5 m straight back: Reeds-Shepp', cp.word(cp.reeds_shepp(S, back, m.RHO)), round(cp.length(cp.reeds_shepp(S, back, m.RHO)), 4),
      '; Dubins', round(cp.length(cp.dubins(S, back, m.RHO)[1]), 4))

sideways 0.5 m to (1.75, 1.433, 0.5236)
  Reeds-Shepp R+ L- R- L+ 1.3181 m, reversals 2 [0.2527, -0.4064, -0.4064, 0.2527]
  Dubins LSL 3.6416 m [0.7854, 0.5, 2.3562]
  times at 0.5 m/s: 2.636 7.283  ratio 2.763
  RO-038 distance between the two poses: 0.5000000000000001
0.5 m straight back: Reeds-Shepp S- 0.5 ; Dubins 3.6416


In [10]:
tries, kept = m.subdivide()
for f0, f1, p, ok in tries:
    print(f'{f0:.2f} -> {f1:.2f}: {cp.word(p):14s} {cp.length(p):.4f} m', 'free' if ok else 'hits an obstacle')
print('pieces', len(kept), 'total', round(sum(cp.length(p) for *_, p in kept), 4), 'm, reversals', cp.cusps([seg for *_, p in kept for seg in p if abs(seg[1]) > 1e-9]))

0.00 -> 1.00: L+ S+ L+       2.9924 m hits an obstacle
0.00 -> 0.50: L+ S+ L+       1.6235 m free
0.50 -> 1.00: R+ S+ L+       1.4613 m hits an obstacle
0.50 -> 0.75: L+ R+ L-       1.1214 m free
0.75 -> 1.00: R+ L+ R-       1.1483 m free
pieces 3 total 3.8932 m, reversals 3


In [11]:
# self-check of the path library: every candidate is driven and must end on the goal; Reeds-Shepp is never longer
# than Dubins, never shorter than the straight line
rng = np.random.default_rng(0)
worst = 0.0
for _ in range(2000):
    st = (rng.uniform(-5, 5), rng.uniform(-5, 5), rng.uniform(-math.pi, math.pi))
    gl = (rng.uniform(-5, 5), rng.uniform(-5, 5), rng.uniform(-math.pi, math.pi))
    rho = rng.uniform(0.3, 3)
    a, b = cp.length(cp.reeds_shepp(st, gl, rho)), cp.length(cp.dubins(st, gl, rho)[1])
    assert a <= b + 1e-9 and a >= math.hypot(gl[0] - st[0], gl[1] - st[1]) - 1e-9
print('2000 random queries passed')

2000 random queries passed


## 8. Kinodynamic RRT

In [12]:
r = m.kino_rrt(seed=4)
print('seed 4: reached', r['reached'], '| iterations', len(r['log']), '| thrown away', sum(not e['ok'] for e in r['log']),
      '| nodes', len(r['nodes']), '| path primitives', len(r['path']) - 1, '| length', r['length'], 'm')
end = r['nodes'][-1]
print('end pose', tuple(round(v, 3) for v in end[:2]), round(math.degrees(end[2]), 1), 'deg; distance to goal',
      round(math.hypot(end[0] - 3.6, end[1] - 3.5), 3))
runs = [m.kino_rrt(seed=s) for s in range(20)]
ok = [x for x in runs if x['reached']]
print('20 seeds: reached', len(ok), '| median nodes', np.median([len(x['nodes']) for x in ok]),
      '| median iterations', np.median([len(x['log']) for x in ok]), '| median path', np.median([x['length'] for x in ok]), 'm')
print('nodes per seed:', [len(x['nodes']) if x['reached'] else None for x in runs])

seed 4: reached True | iterations 221 | thrown away 147 | nodes 75 | path primitives 8 | length 3.0 m
end pose (np.float64(3.41), np.float64(3.35)) 73.0 deg; distance to goal 0.243


20 seeds: reached 18 | median nodes 76.5 | median iterations 200.0 | median path 3.375 m
nodes per seed: [21, 27, 2043, 51, 75, None, None, 2419, 24, 2552, 62, 18, 2135, 23, 78, 439, 3960, 22, 453, 325]
